In [2]:
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from plotly.subplots import make_subplots
import plotly.graph_objs as go
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, make_scorer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import RandomizedSearchCV
import shap
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
import plotly.express as px
from catboost import CatBoostClassifier

In [3]:
RANDOM_STATE = 12345

In [4]:
df = pd.read_csv("../data/adult.csv")

Разделение на X и у

In [5]:
X = df.drop('income', axis = 1)

In [6]:
y = df['income']

In [7]:
X.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States


In [8]:
y.value_counts()

income
<=50K    37155
>50K     11687
Name: count, dtype: int64

кодирование целевого признака

In [9]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(label_encoder.classes_)

['<=50K' '>50K']


In [10]:
list(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

[('<=50K', np.int64(0)), ('>50K', np.int64(1))]

Разделение на тренировочную и тестовую выборки


In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size = 0.2, random_state = RANDOM_STATE, stratify=y_encoded)

In [12]:
X_train.shape

(39073, 14)

In [13]:
X_test.shape

(9769, 14)

Разделелили согласно условию

In [14]:
num_cols = X_train.select_dtypes(include='number').columns.tolist()

In [15]:
cat_cols = X_train.select_dtypes(include='object').columns.tolist()

/var/folders/jt/pfrf2kqn59d1ph0418pyq4mr0000gn/T/ipykernel_7073/4180746908.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include='object').columns.tolist()


In [16]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

In [17]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "CatBoost": CatBoostClassifier(
        iterations=300,
        depth=6,
        learning_rate=0.1,
        verbose=0,
        random_state=42
    )
}

In [18]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

baseline_results = []

for model_name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )
    
    baseline_results.append({
        "model": model_name,
        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
        "precision_mean": scores["test_precision"].mean(),
        "recall_mean": scores["test_recall"].mean(),
        "f1_mean": scores["test_f1"].mean(),
        "roc_auc_mean": scores["test_roc_auc"].mean(),
        "roc_auc_std": scores["test_roc_auc"].std()
    })

baseline_results_df = (
    pd.DataFrame(baseline_results)
    .sort_values("roc_auc_mean", ascending=False)
    .reset_index(drop=True)
)

display(baseline_results_df)

/Users/aleksejtolkunov/Desktop/my_project/Classification_python/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/aleksejtolkunov/Desktop/my_project/Classification_python/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also

,model,accuracy_mean,accuracy_std,precision_mean,recall_mean,f1_mean,roc_auc_mean,roc_auc_std
0,CatBoost,0.874363,0.001060,0.790518,0.646272,0.711094,0.928478,0.001955
1,Random Forest,0.853454,0.002766,0.728083,0.619104,0.669042,0.902289,0.002540
2,Logistic Regression,0.845264,0.004351,0.716830,0.584124,0.643622,0.890537,0.006849
3,Decision Tree,0.813042,0.001897,0.606511,0.623061,0.614612,0.747969,0.001329


По результатам подбора параметров лучшие результаты показывает модель CatBoost f1_mean 0.711094, roc_auc_mean 0.928478. Протестируем лучшую модель на тестовых данных	

In [19]:
best_model_name = baseline_results_df.loc[0, "model"]
best_model_name

best_model = models[best_model_name]

best_baseline_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_model)
])

best_baseline_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [20]:
y_pred = best_baseline_pipeline.predict(X_test)
y_proba = best_baseline_pipeline.predict_proba(X_test)[:, 1]

test_metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_proba)
}

test_metrics_df = pd.DataFrame([test_metrics])
display(test_metrics_df)

,accuracy,precision,recall,f1,roc_auc
0,0.872351,0.778743,0.651839,0.709662,0.930553


Построим матрицу ошибок

In [21]:
cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=[f"actual_{cls}" for cls in label_encoder.classes_],
    columns=[f"pred_{cls}" for cls in label_encoder.classes_]
)

display(cm_df)

,pred_<=50K,pred_>50K
actual_<=50K,6998,433
actual_>50K,814,1524


Построим график важности признаков

In [25]:
catboost_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", CatBoostClassifier(
        iterations=300,
        depth=6,
        learning_rate=0.1,
        verbose=0,
        random_state=42
    ))
])

catboost_pipeline.fit(X_train, y_train)

feature_names = (
    catboost_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

importances = (
    catboost_pipeline
    .named_steps["model"]
    .feature_importances_
)

feature_importance_df = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(feature_importance_df.head(20))

,feature,importance
0,num__capital-gain,21.845554
1,cat__marital-status_Married-civ-spouse,21.797092
2,num__age,11.554919
3,num__educational-num,9.270241
4,num__capital-loss,7.918638
5,num__hours-per-week,7.405039
6,num__fnlwgt,2.450500
7,cat__relationship_Husband,1.460427
8,cat__occupation_Exec-managerial,1.401991
9,cat__occupation_Other-service,1.299427


In [26]:
top_features = feature_importance_df.head(20).sort_values("importance")

fig = px.bar(
    top_features,
    x="importance",
    y="feature",
    orientation="h",
    title="Топ-20 признаков по важности для CatBoost"
)

fig.update_layout(
    xaxis_title="Важность признака",
    yaxis_title="Признак"
)

fig.show()

По графику видно, что наиболее важными признаками семейное положение, образование, количество раб часов в неделю.